# Assignment 6: Neural Network Showdown — Solution

## Setup

In [1]:
%pip install -q -r requirements.txt

# GPU acceleration (platform-specific)
import platform
if platform.system() == "Darwin" and platform.machine() == "arm64":
    %pip install -q tensorflow-metal

%reset -f

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import numpy as np
import pandas as pd

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

# Report available accelerators
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU acceleration: {len(gpus)} device(s)")
    for gpu in gpus:
        print(f"  {gpu.name}")
else:
    print("No GPU detected — using CPU")

from tensorflow import keras
from keras import Sequential
from keras.layers import (
    Dense, Flatten, Dropout, Conv2D, MaxPooling2D, LSTM, Input
)
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import confusion_matrix

from helpers import (
    load_cifar10, load_ecg5000,
    plot_training_history, plot_confusion_matrix,
    CIFAR10_CLASSES, ECG_CLASSES,
)

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete!")

GPU acceleration: 1 device(s)
  /physical_device:GPU:0
Setup complete!


---

## Part 1: Dense Baseline on CIFAR-10

In [3]:
print("Part 1: Dense Baseline on CIFAR-10")
print("-" * 40)

X_train, y_train, X_test, y_test = load_cifar10()
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Part 1: Dense Baseline on CIFAR-10
----------------------------------------


Train: (50000, 32, 32, 3), Test: (10000, 32, 32, 3)


In [4]:
model_dense = Sequential([
    Input(shape=(32, 32, 3)),
    Flatten(),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(10, activation="softmax"),
])

In [5]:
model_dense.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [6]:
early_stop = EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history_dense = model_dense.fit(
    X_train, y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
)

Epoch 1/20


  1/352 ━━━━━━━━━━━━━━━━━━━━ 6:38 1s/step - accuracy: 0.1250 - loss: 2.9133

  4/352 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.1245 - loss: 5.6203

  7/352 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.1207 - loss: 6.5603

 10/352 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.1209 - loss: 7.2241

...

335/352 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2594 - loss: 2.1014

340/352 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2594 - loss: 2.1013

345/352 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2594 - loss: 2.1012

350/352 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2594 - loss: 2.1011

352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.2607 - loss: 2.0943 - val_accuracy: 0.2880 - val_loss: 1.9899


In [7]:
test_loss, test_acc = model_dense.evaluate(X_test, y_test, verbose=0)

y_pred = np.argmax(model_dense.predict(X_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
cm = confusion_matrix(y_true, y_pred)

In [8]:
results = {
    "accuracy": float(test_acc),
    "confusion_matrix": cm.tolist(),
}
with open(os.path.join(OUTPUT_DIR, "part1_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"Dense accuracy: {test_acc:.4f}")
print("Saved output/part1_results.json")

Dense accuracy: 0.2839
Saved output/part1_results.json


---

## Part 2: CNN on CIFAR-10

In [9]:
print("\nPart 2: CNN on CIFAR-10")
print("-" * 40)


Part 2: CNN on CIFAR-10
----------------------------------------


In [10]:
model_cnn = Sequential([
    Input(shape=(32, 32, 3)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    Flatten(),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax"),
])

In [11]:
model_cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [12]:
callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    ModelCheckpoint(
        "output/best_cnn.keras",
        save_best_only=True,
        monitor="val_accuracy",
    ),
]

history_cnn = model_cnn.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks,
)

Epoch 1/15


  1/704 ━━━━━━━━━━━━━━━━━━━━ 20:20 2s/step - accuracy: 0.1094 - loss: 2.3335

  4/704 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.0921 - loss: 2.3275

  8/704 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.0904 - loss: 2.3200

 12/704 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.0920 - loss: 2.3186

...

689/704 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4251 - loss: 2.3740

693/704 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4250 - loss: 2.3777

697/704 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4249 - loss: 2.3815

701/704 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4249 - loss: 2.3853

704/704 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.4106 - loss: 3.0655 - val_accuracy: 0.3628 - val_loss: 4.9590


In [13]:
plot_training_history(
    history_cnn, os.path.join(OUTPUT_DIR, "part2_training_history.png")
)

In [14]:
cnn_loss, cnn_acc = model_cnn.evaluate(X_test, y_test, verbose=0)

y_pred_cnn = np.argmax(model_cnn.predict(X_test, verbose=0), axis=1)
y_true_cnn = np.argmax(y_test, axis=1)
cm_cnn = confusion_matrix(y_true_cnn, y_pred_cnn)

In [15]:
results_cnn = {
    "accuracy": float(cnn_acc),
    "confusion_matrix": cm_cnn.tolist(),
}
with open(os.path.join(OUTPUT_DIR, "part2_results.json"), "w") as f:
    json.dump(results_cnn, f, indent=2)

comparison = pd.DataFrame([
    {"model": "Dense", "accuracy": float(test_acc)},
    {"model": "CNN", "accuracy": float(cnn_acc)},
])
comparison.to_csv(os.path.join(OUTPUT_DIR, "part2_comparison.csv"), index=False)

print(f"CNN accuracy:   {cnn_acc:.4f}")
print(f"Dense accuracy: {test_acc:.4f}")
print(f"Improvement:    {cnn_acc - test_acc:+.4f}")
print("Saved output/part2_results.json and output/part2_comparison.csv")

CNN accuracy:   0.5718
Dense accuracy: 0.2839
Improvement:    +0.2879
Saved output/part2_results.json and output/part2_comparison.csv


---

## Part 3: LSTM on ECG5000

In [16]:
print("\nPart 3: LSTM on ECG5000")
print("-" * 40)

X_train_ecg, y_train_ecg, X_test_ecg, y_test_ecg = load_ecg5000()
print(f"Train: {X_train_ecg.shape}, Test: {X_test_ecg.shape}")
print(f"Classes: {list(ECG_CLASSES.values())}")


Part 3: LSTM on ECG5000
----------------------------------------
Train: (4000, 140, 1), Test: (1000, 140, 1)
Classes: ['Normal', 'Supraventricular', 'Premature Ventricular', 'Fusion', 'Unknown']


In [17]:
model_lstm = Sequential([
    Input(shape=(140, 1)),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(5, activation="softmax"),
])

In [18]:
model_lstm.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [19]:
early_stop = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history_lstm = model_lstm.fit(
    X_train_ecg, y_train_ecg,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
)

Epoch 1/30


  1/113 ━━━━━━━━━━━━━━━━━━━━ 3:31 2s/step - accuracy: 0.0625 - loss: 1.6509

  4/113 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.1510 - loss: 1.6220

  8/113 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.2607 - loss: 1.5781

 11/113 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.3313 - loss: 1.5444

...

102/113 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9585 - loss: 0.1415

105/113 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9584 - loss: 0.1417

108/113 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9584 - loss: 0.1418

112/113 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9583 - loss: 0.1421

113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9550 - loss: 0.1498 - val_accuracy: 0.9275 - val_loss: 0.2082


In [20]:
plot_training_history(
    history_lstm, os.path.join(OUTPUT_DIR, "part3_training_history.png")
)

In [21]:
lstm_loss, lstm_acc = model_lstm.evaluate(X_test_ecg, y_test_ecg, verbose=0)

y_pred_ecg = np.argmax(model_lstm.predict(X_test_ecg, verbose=0), axis=1)
y_true_ecg = np.argmax(y_test_ecg, axis=1)
cm_ecg = confusion_matrix(y_true_ecg, y_pred_ecg)

In [22]:
results_ecg = {
    "accuracy": float(lstm_acc),
    "confusion_matrix": cm_ecg.tolist(),
}
with open(os.path.join(OUTPUT_DIR, "part3_results.json"), "w") as f:
    json.dump(results_ecg, f, indent=2)

print(f"LSTM accuracy: {lstm_acc:.4f}")
print("Saved output/part3_results.json")

LSTM accuracy: 0.9370
Saved output/part3_results.json


---

## Validation

In [23]:
print("\nAll parts complete!")
print("Run 'pytest .github/tests/ -v' in your terminal to check your work.")


All parts complete!
Run 'pytest .github/tests/ -v' in your terminal to check your work.
